In [2]:
import cv2
import mediapipe as mp
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional,LSTM, Dense,Input,Flatten, GRU

from glob import glob
import os
from pathlib import Path

from tqdm.notebook import tqdm

from numpy import  argmax, array, expand_dims 
import numpy as np

import utils


In [3]:
# Initialize Mediapipe Pose model
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils  
# pose = mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7)


mp_utils = utils.MediapipeUtils(mp_pose, mp_drawing)

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [5]:
# Specify the video file(s)
DATA_DIR = glob("Dataset/*/*/Augmentation/*.mp4")
DATA_PATH = "MP_Data"

# Landmarks to extract (based on Mediapipe Pose Landmarks)
# landmark_indices = [11, 12, 13, 14, 15, 16]

angle_landmarks = ((12,14),
                   (14,16),
                   (11,13),
                   (13,15))

labels = [i.split("\\")[-1] for i in glob("Dataset\*\*")]

DATA_DIR,labels

(['Dataset\\Adjectives\\Blind\\Augmentation\\1-2_centered.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-2_downsampled.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-3_centered.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-3_downsampled.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-4_centered.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-4_downsampled.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-5_centered.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-5_downsampled.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-6_centered.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-6_downsampled.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-7_centered.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-7_downsampled.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-8_centered.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\1-8_downsampled.mp4',
  'Dataset\\Adjectives\\Blind\\Augmentation\\Augmenta_centered.mp4',
  'Dataset\\Ad

In [4]:
try:
    # os.mkdir('MP_data')
    for i in range(len(labels)):
        os.mkdir(f'MP_data/{labels[i]}')
except:
    print("Directory already exists")

Directory already exists


In [9]:
import importlib

importlib.reload(utils)

# Process each video
for video_path in tqdm(DATA_DIR):
    data = []
    cap = cv2.VideoCapture(video_path)
    frame_count = 0

        
    with mp_pose.Pose(min_detection_confidence=0.7,  
                            min_tracking_confidence=0.7) as pose:
    
        while cap.isOpened():

            ret, frame = cap.read()
            if not ret:
                break
                        
            image, results = mp_utils.mediapipe_detection(frame, pose)

            mp_utils.draw_styled_landmarks(image, results)

            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                
                features = mp_utils.extract_features(landmarks)

                data.append(features)    
            
            else:
                data.append(np.zeros(8))

        data = np.array(data)
        # print(data.shape)
        
        path = video_path.split("/")
        
        # Getting Current action of the video
        action = video_path.split("\\")[-3]
        
        #Get the sequence number
        sequence = video_path.split("\\")[-1][:-4]
        
        npy_path = os.path.join(DATA_PATH, action, sequence)
        # print(npy_path)
        
        np.save(npy_path, data)
        cap.release()



  0%|          | 0/220 [00:00<?, ?it/s]